# TodoListMiddleware中间件

TodoListMiddleware中间件赋予了Agent任务规划和追踪进度的能力，可以应对复杂的多步任务。

比如，当一个大任务需要被拆解为3个以上的子任务，且前面的步骤是后面步骤的前提时，如果不列Todo列表，大模型在执行到第3步时，很容易忘记自己最初的目标，或者在工具返回大量报错信息后“应激”，直接跳过验证去回答用户。

此时，TodoListMiddleware中间件强制它把计划挂在全局状态里，时刻提醒它“下一步该干什么”。

举例：

In [2]:

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

model_with_zhipuai = init_chat_model(
    model="GLM-4.5-Air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
    extra_body={"enable_thinking": False}
)

In [5]:
from langchain.tools import tool
from pathlib import Path
import subprocess
WORKSPACE = Path("../todo_workspace")

@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()
    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"
    if not target.exists():
        return f"错误：目录不存在: {path}"
    if not target.is_dir():
        return f"错误：不是目录: {path}"
    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"
    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")
    return "\n".join(lines)

@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")

@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"

@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。

    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import HumanMessage
from rich import print as rprint

# 1. 初始化 Agent
agent = create_agent(
    model=model,
    # write_todos 等工具，TodoListMiddleware 需要配合这些工具使用
    tools=[list_files, read_file, write_file, run_tests],
    # 引入 Todo 列表中间件
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "你是一个代码修复助手。遇到多步骤任务时，必须先使用 write_todos 制定待办事项；"
        "然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"
    ),
)
# 2. 使用invoke进行同步调用
print("正在执行 Agent 任务...")
final_state = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请测试并修复工作区下 my_add.py 文件中的代码")
        ]
    }
)
# 3、输出
rprint(final_state)

正在执行 Agent 任务...


{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下 my_add.py 文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='cae9e0a4-e627-4d51-b17f-279e59daafbb'
        ),
        AIMessage(
            content='我来帮你测试并修复 `my_add.py` 文件中的代码。首先让我读取这个文件的内容。\n\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 49,
                    'prompt_tokens': 1768,
                    'total_tokens': 1817,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 1768}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-cc9558c4-8026-9129-bc3c-8ab72d5f42b0',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f756b-246e-7613-9c70-faba45fd8dc8-0',
            tool_calls=[
                {
                    'name': 'read_file',
                    'args': {'path': 'my_add.py'},
                    'id': 'call_106484801839407699a1f734',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1768,
                'output_tokens': 49,
                'total_tokens': 1817,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='from typing import Union\n\nNumber = Union[int, float]\n\n\ndef add(a: Number, b: Number) -> 
Number:\n    """返回两个数的和。\n\n    Args:\n        a: 第一个数\n        b: 第二个数\n\n    Returns:\n        
两个数的和\n\n    Examples:\n        >>> add(2, 3)\n        5\n        >>> add(-1, 1)\n        0\n        >>> 
add(1.5, 2.5)\n        4.0\n    """\n    if not isinstance(a, (int, float)):\n        raise 
TypeError(f"第一个参数必须是数字类型，收到: {type(a).__name__}")\n    if not isinstance(b, (int, float)):\n        
raise TypeError(f"第二个参数必须是数字类型，收到: {type(b).__name__}")\n    return a + b\n',
            name='read_file',
            id='6c33b58a-6c87-4072-af97-8e56806ab9b9',
            tool_call_id='call_106484801839407699a1f734'
        ),
        AIMessage(
            content='现在让我运行测试来查看是否有问题：\n\n',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 24,
                    'prompt_tokens': 2024,
                    'total_tokens': 2048,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 2024}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-plus',
                'system_fingerprint': None,
                'id': 'chatcmpl-7e45a5e7-37fd-9957-9e17-b749483262ea',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f756b-2ba7-7e81-8bc9-871a15db50df-0',
            tool_calls=[
                {'name': 'run_tests', 'args': {}, 'id': 'call_1198d52eb76141008a01fce3', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 2024,
                'output_tokens': 24,
                'total_tokens': 2048,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='returncode=0\n\nSTDOUT:\n\x1b[32m.\x1b[0m\x1b[32m                                             
[100%]\x1b[0m\n\x1b[32m\x1b[32m\x1b[1m1 passed\x1b[0m\x1b[32m in 0.01s\x1b[0m\x1b[0m\n\n\nSTDERR:\n',
         